In [7]:
pip install ollama

In [8]:
!pip install ollama

In [11]:
import json
import docx
import ollama

# Read the document file
def read_word_file(file_path):
    doc = docx.Document(file_path)
    full_text = []
    for paragraph in doc.paragraphs:
        full_text.append(paragraph.text)
    return '\n'.join(full_text)


# This remains a raw DICT, but booleans in JSON (False/True) must be Python tokens (False/True)
def generate_expected_json_template():
    template = {
        "type": "object",
        "additionalProperties": False,
        "required": [
            "stage1_ingestion",
            "catalyst",
            "actor_characteristics",
            "attack_characteristics",
            "organisation_characteristics",
        ],
        "properties": {
            "stage1_ingestion": {
                "type": "object",
                "additionalProperties": False,
                "required": [
                    "case_id",
                    "source",
                    "reference",
                    "case_summary",
                    "include_for_stage_2",
                    "screening_note",
                ],
                "properties": {
                    "case_id": {
                        "type": "string",
                        "pattern": "^CS-[0-9]{3,}$",
                        "description": "Unique case identifier (e.g., CS-001).",
                    },
                    "source": {
                        "type": "string",
                        "description": "Source of the case data.",
                    },
                    "reference": {
                        "type": "string",
                        "description": "Unique case reference, citation, URL, or document name.",
                    },
                    "case_summary": {
                        "type": "string",
                        "description": "Short factual summary of the case.",
                    },
                    "include_for_stage_2": {
                        "type": "boolean",
                        "description": "Whether the case proceeds to Stage 2 analysis.",
                    },
                    "screening_note": {
                        "type": "string",
                        "description": "Analyst screening notes.",
                    },
                },
            },
            "catalyst": {
                "type": "object",
                "additionalProperties": False,
                "required": ["precipitating_event"],
                "properties": {
                    "precipitating_event": {
                        "type": "array",
                        "description": "Triggering event or catalyst associated with the insider behaviour.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "employee_dismissal",
                                "demotion",
                                "resignation",
                                "disputes_with_employers",
                                "perceived_injustices",
                                "negative_company_acts",
                                "family_problems",
                                "coercion",
                                "new_opportunities",
                                "other",
                                "unknown",
                            ],
                        },
                    }
                },
            },
            "actor_characteristics": {
                "type": "object",
                "additionalProperties": False,
                "required": [
                    "actors",
                    "psychological_state",
                    "personality_characteristics",
                    "attitude_towards_work",
                    "motivation_to_attack",
                    "skill_set",
                    "opportunity",
                    "historical_behaviour",
                    "observed_physical_behaviour",
                    "observed_cyber_behaviour",
                ],
                "properties": {
                    "actors": {
                        "type": "array",
                        "description": "Actor-specific information. Add one object for each actor involved in the incident.",
                        "items": {
                            "type": "object",
                            "additionalProperties": False,
                            "required": [
                                "actor_id",
                                "type_of_actor",
                                "enterprise_role",
                                "state_of_relationship",
                            ],
                            "properties": {
                                "actor_id": {
                                    "type": "string",
                                    "description": "Identifier for the actor (e.g., actor_1, actor_2).",
                                },
                                "type_of_actor": {
                                    "type": "string",
                                    "enum": [
                                        "employee",
                                        "former_employee",
                                        "contractor",
                                        "business_partner",
                                        "other",
                                        "unknown",
                                    ],
                                },
                                "enterprise_role": {
                                    "type": "string",
                                    "enum": [
                                        "scientist",
                                        "engineer",
                                        "programmer",
                                        "salesperson",
                                        "manager",
                                        "security_person",
                                        "finance_person",
                                        "executive",
                                        "support_person",
                                        "military_person",
                                        "other",
                                        "unknown",
                                    ],
                                },
                                "state_of_relationship": {
                                    "type": "string",
                                    "enum": [
                                        "current",
                                        "former",
                                        "serving_notice",
                                        "temporary",
                                        "contract",
                                        "other",
                                        "unknown",
                                    ],
                                },
                            },
                        },
                    },
                    "psychological_state": {
                        "type": "array",
                        "description": "Actor's psychological or emotional state.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "financial_stress",
                                "grievances",
                                "misjudgement",
                                "rationalisation",
                                "disgruntlement",
                                "aggression",
                                "job_dissatisfaction",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "personality_characteristics": {
                        "type": "array",
                        "description": "Actor personality characteristics.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "greed",
                                "selfishness",
                                "personal_inflexibilities",
                                "narcissism",
                                "psychopathy",
                                "low_agreeableness",
                                "risk_tolerance",
                                "honesty_humility",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "attitude_towards_work": {
                        "type": "array",
                        "description": "Actor's feelings or behavioural orientation toward work.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "dissatisfied",
                                "disengaged",
                                "resentful",
                                "hostile",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "motivation_to_attack": {
                        "type": "array",
                        "description": "Reason or motive for the attack.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "financial_reward",
                                "personal_gain",
                                "revenge",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "skill_set": {
                        "type": "array",
                        "description": "Description of the actor's relevant skills or capabilities.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "technical_expertise",
                                "privileged_system_knowledge",
                                "data_exploitation_capability",
                                "deception/social_engineering_capability",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "opportunity": {
                        "type": "array",
                        "description": "Actor's opportunity to initiate or conduct the attack.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "privileged_access",
                                "physical_access",
                                "remote_access",
                                "shared_credentials",
                                "no_access_revocation",
                                "insufficient_monitoring",
                                "trusted_position",
                                "third_party_access",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "historical_behaviour": {
                        "type": "array",
                        "description": "Relevant historical behaviour before the incident.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "addictive_practices",
                                "harassment",
                                "company_policy_violation",
                                "criminal_history",
                                "history_of_serious_mental_problems",
                                "carelessness",
                                "absent_mindedness",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "observed_physical_behaviour": {
                        "type": "array",
                        "description": "Observed physical-world behaviour relevant to the case.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "poor_work_performance",
                                "workplace_rule_violation",
                                "conflicts_with_colleagues",
                                "increased_outbursts",
                                "substance_abuse",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "observed_cyber_behaviour": {
                        "type": "array",
                        "description": "List of observed cyber activities.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "access_and_authentication",
                                "data_and_file_management",
                                "communication_activities",
                                "system_and_network_behaviour",
                                "privilege_and_security_behaviour",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                },
            },
            "attack_characteristics": {
                "type": "object",
                "additionalProperties": False,
                "required": [
                    "attack",
                    "attack_objective",
                    "attack_step",
                    "attack_step_goal",
                ],
                "properties": {
                    "attack": {
                        "type": "array",
                        "description": "Whether the harmful activity was intentional, unintentional, other, or unknown.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "intentional",
                                "unintentional",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "attack_objective": {
                        "type": "array",
                        "description": "Primary objective or outcome associated with the attack.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "data_theft",
                                "sabotage",
                                "fraud",
                                "espionage",
                                "mixed",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "attack_step": {
                        "type": "array",
                        "description": "Specific activities undertaken to conduct the attack.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "preparation/planning",
                                "target_identification",
                                "information_gathering /Discovery",
                                "access/access_acquisition",
                                "privilege/permission_Abuse",
                                "collection/acquisition",
                                "execution/action",
                                "data_transfer/exfiltration",
                                "manipulation/modification",
                                "sabotage/disruption",
                                "concealment/evasion",
                                "persistence/continued_access",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "attack_step_goal": {
                        "type": "array",
                        "description": "Specific outcome/goal after the attack steps take place",
                        "items": {
                            "type": "string",
                            "enum": [
                                "confidentiality_loss",
                                "integrity_loss",
                                "availability_loss",
                                "financial_loss",
                                "reputational_damage",
                                "privacy_loss",
                                ":operational_disruption",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                },
            },
            "organisation_characteristics": {
                "type": "object",
                "additionalProperties": False,
                "required": ["asset", "vulnerability"],
                "properties": {
                    "asset": {
                        "type": "array",
                        "description": "Targeted organisational asset, data, system, process, or service.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "information_asset",
                                "technology_asset",
                                "organisational_asset",
                                "intangible_asset",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "vulnerability": {
                        "type": "array",
                        "description": "Organisational or control vulnerabilities that enabled or worsened the incident.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "lack_of_security_policies",
                                "inconsistent_security_policies",
                                "poor_security_practices",
                                "organisational_culture",
                                "lack_of_support",
                                "poor_communication",
                                "poor_management_practices",
                                "leadership_issues",
                                "insufficient_monitoring",
                                "lack_of_oversight",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                },
            },
        },
    }
    return template
    

# Task:
# Analyse the following case study and extract information into the requested schema fields. Use the rules defined in the Coding Guide.
# When picking values for attack steps, conside MITRE Attack faremework https://attack.mitre.org/. Values in the framework should be exactly the same in the attack steps. Mention it with the tactic Id



# Prompts can be more concise because the schema structure handles enforcement
def build_prompt(case_text, coding_guide_text, case_id):
    return f"""
### Role

You are an expert cybersecurity analyst specializing in insider threat analysis. Your task is to analyse an insider threat case study and extract information according to the provided coding schema and Coding Guide.

### Instructions

1. Carefully read the case study and identify information that explicitly appears in the text.

2.The source case ID is: {case_id}
  You MUST use this exact value for: stage1_ingestion.case_id
  Do not generate, infer, or change the case ID.

3. Populate all fields in the provided JSON schema following the definitions and rules in the Coding Guide.

4. Use only the values defined in the Coding Guide for categorical fields. Do not create new categories or modify existing labels.


### Output Requirements

Return only the completed JSON object following the provided schema.

Coding Guide:
{coding_guide_text}

Case Study:
{case_text}
"""
coding_guide_path = "C:/Users/Nadeeka_92/Downloads/Coding_Manual_v2_2_updated.docx"
guide_text = read_word_file(coding_guide_path)

def call_ollama(prompt, model_name="llama3.1"):
    """
    Calls local Ollama server and enforces the structured JSON schema.
    Recommended models: 'mistral', 'llama3.1', or 'qwen2.5'
    """
    json_schema = generate_expected_json_template()

    # Call Ollama using the official library
    response = ollama.chat(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        format=json_schema,  # Ollama accepts raw JSON schema dict directly
        options={
            "temperature": 0  # Set temperature for deterministic output
        }
    )

    # Extract text and parse as JSON dict
    response_content = response["message"]["content"]
    return json.loads(response_content)


# --- Execution Example ---
coding_guide_path = "C:/Users/Nadeeka_92/Downloads/coding_guide.docx"
guide_text = read_word_file(coding_guide_path)
case_text = "Your case study text here..."  # Replace with actual case study loading logic

# -commented Janaka all lines below
# prompt = build_prompt(case_text, guide_text)
# result = call_ollama(prompt, model_name="mistral")

# print(json.dumps(result, indent=2))

In [ ]:
import json
import os
import pandas as pd
import time
import docx
import ollama

# --- Helper Functions ---

def read_word_file(file_path):
    """Reads the coding guide Word document."""
    doc = docx.Document(file_path)
    full_text = [paragraph.text for paragraph in doc.paragraphs]
    return '\n'.join(full_text)

def call_ollama(prompt, model_name="llama3.1"):
    """
    Calls local Ollama server and enforces the JSON Schema format.
    Other models that I can try for complex schema compliance: 'llama3.1', 'qwen2.5', or 'mistral'.
    """
    json_schema = generate_expected_json_template()

    # Pass the Python schema dictionary directly to format
    response = ollama.chat(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        format=json_schema,
        options={
            "temperature": 0  # Deterministic output for structured extraction
        }
    )

    response_content = response["message"]["content"]
    return json.loads(response_content)


# --- Main Execution Loop ---

# 1. Load the Coding Guide
coding_guide_path = "C:/Users/Nadeeka_92/Downloads/coding_guide.docx"
guide_text = read_word_file(coding_guide_path)

# 2. Create dedicated output directory
output_dir = "Case_Study_JSONs_Ollama"
os.makedirs(output_dir, exist_ok=True)

# 3. Load dataset
csv_path = "casestest123.csv"
df = pd.read_csv(csv_path, encoding="latin1")

print(f"Loaded {len(df)} cases from CSV. Starting local extraction via Ollama...\n")

# Select your local model name ('llama3.1', 'qwen2.5', 'mistral', etc.)
OLLAMA_MODEL = "llama3.1"

# 4. Iterate through each row in the CSV file
for index, row in df.iterrows():
    case_text = row.get("description")
    
    # Skip empty rows
    if pd.isna(case_text) or not str(case_text).strip():
        print(f"Row {index}: Skipped (No description found)")
        continue

    output_filename = os.path.join(output_dir, f"CS_{index:03d}_LLM.json")
    
    # Skip if already processed to allow resuming
    if os.path.exists(output_filename):
        print(f"Row {index}: Already processed. Skipping...")
        continue

    print(f"Processing Row {index} with Ollama ({OLLAMA_MODEL})...")

    try:
        # Build prompt using current row text and guide
        prompt = build_prompt(case_text, guide_text, row)

        # Call local Ollama model
        output_dict = call_ollama(prompt, model_name=OLLAMA_MODEL)

        # Save output to individual JSON
        with open(output_filename, "w", encoding="utf-8") as f:
            json.dump(output_dict, f, indent=2, ensure_ascii=False)

        print(f" ✅ Success! Saved to {output_filename}")

    except Exception as e:
        print(f" ❌ Error on Row {index}: {e}")

    # Optional short pause between local GPU/CPU runs (reduced from 5s to 1s since local API rate limits don't apply)
    time.sleep(1)

print("\n🎉 All case studies have been processed successfully!")

Loaded 5 cases from CSV. Starting local extraction via Ollama...

Processing Row 0 with Ollama (llama3.1)...
